# Data Manipulation — Basic
## Data Cleaning & Quality Control (R)

**Philosophy:** This note is pattern-first, not syntax-first. It assumes you know
the tidyverse API. Each section answers: *you have data that looks like X — here
is the full pipeline to fix it.*

---

## Decision Table

| If your data has... | Go to |
| :--- | :--- |
| Unknown shape, types, or quality | §1 — Audit |
| Dates as strings, IDs as doubles, booleans as character | §2 — Fix types |
| Missing values and unsure what to do | §3 — Missing Values |
| Suspected duplicate rows | §4 — Duplicates |
| Inconsistent category labels (`"US"`, `"us"`, `" USA "`) | §5 — String & Categorical |
| Extreme values distorting results | §6 — Outliers |
| Column names with spaces, caps, or special characters | §7 — Column Names |

## Series Map

| Notebook | Use it for |
| :--- | :--- |
| `DM_Basic` **(this note)** | Cleaning & QC — audit, types, nulls, dupes, strings, outliers, column names |
| `DM_Intermediate` | Reshaping & combining — wide/long, group-by, multi-table joins, time-series, nested, many files |
| `DM_Advanced` | Feature engineering — encoding, transforms, binning, date, lag/window, interactions, text, leakage |
| `Reference_Tidyverse / Reference_BaseR` | Syntax lookups for individual functions |

> **Environment:** R 4.x / tidyverse 2.x. dplyr verbs always return a **new** object
> — there is no `inplace=True`-style trap and no copy-on-write subtlety to manage;
> every `df <- df |> mutate(...)` is already explicit and safe. A sample frame is
> defined below so the patterns are runnable.

In [ ]:
library(dplyr)
library(tidyr)
library(stringr)
library(lubridate)

# Sample messy frame to run the cleaning patterns against.
# Replace with your own:  df <- readr::read_csv('data.csv')
df <- tibble::tibble(
  user_id    = c(1, 2, 3, 4, 4, 5),
  event_date = c('2024-01-05','2024-01-06','2024/01/07','08-01-2024','08-01-2024', NA),
  age        = c(34, 28, -5, 41, 41, 130),
  revenue    = c('$1,200', '950', '', '2,000', '2,000', '15'),
  country    = c('US', ' us ', 'USA', 'Canada', 'Canada', 'cnaada'),
  platform   = c('iOS','Android','Web','iOS','iOS','Web')
)
print(df)

---
## §1 — Audit a New Dataset

Run this sequence every time you load a new dataset — before touching anything.
The goal is to build a complete picture of what you have and what needs fixing.

In [ ]:
library(dplyr)
library(readr)

df <- read_csv('data.csv')

# ── Step 1: Shape and preview ───────────────────────────────────────────────
dim(df)                  # How many rows and columns?
head(df)                 # Does the data look as expected?
tail(df)                 # Does the end of the file look clean (no trailing junk rows)?

# ── Step 2: Types and nulls ─────────────────────────────────────────────────
glimpse(df)               # Types + a row preview per column -- the fastest signal of type problems

# ── Step 3: Null summary ────────────────────────────────────────────────────
null_summary <- tibble::tibble(
  column     = names(df),
  null_count = colSums(is.na(df)),
  null_rate  = round(colMeans(is.na(df)), 3)
) |> filter(null_count > 0) |> arrange(desc(null_rate))
print(null_summary)
# Interpretation:
# null_rate < 0.05  -> likely safe to drop or fill
# null_rate > 0.50  -> question whether the column is usable at all

# ── Step 4: Duplicate check ─────────────────────────────────────────────────
cat('Full duplicates:', sum(duplicated(df)), '\n')
cat('Key-level duplicates:', sum(duplicated(df[c('user_id', 'date')])), '\n')

# ── Step 5: Cardinality and distributions ───────────────────────────────────
summary(select(df, where(is.numeric)))                  # Numeric: spot impossible min/max
summary(select(df, where(is.character)))                 # Categorical: top value and frequency

char_cols <- names(df)[sapply(df, is.character)]
for (col in char_cols) {
  vals <- df[[col]]
  top3 <- sort(table(vals), decreasing = TRUE)[1:3]
  cat(sprintf("%s: %d unique -- %s\n", col, n_distinct(vals), paste(names(top3), top3, sep = ":", collapse = ", ")))
}
# Look for: unexpectedly high cardinality (ID masquerading as category)
# and low cardinality numerics that should be categorical

# ── Step 6: Value range sanity checks ──────────────────────────────────────
cat(min(df$age, na.rm = TRUE), max(df$age, na.rm = TRUE), '\n')   # Negative age? Age > 120?
cat(min(df$revenue, na.rm = TRUE), '\n')                           # Negative revenue -- refund or error?
cat(sum(df$rate > 1, na.rm = TRUE), '\n')                          # Rate column -- should be 0-1 or 0-100?

**What each step is looking for:**

| Step | Red flags to catch |
| :--- | :--- |
| Shape + preview | Extra header rows, merged cells exported from Excel, trailing commas |
| `glimpse()` | Date columns as `character`, ID columns as `double` (signals nulls present), boolean as `chr`/`int` |
| Null summary | Columns with >50% nulls — question if they're usable |
| Duplicates | Full duplicates = data pipeline bug; key-level = fan-out from a join |
| Cardinality | String columns with thousands of unique values — probably IDs, not categories |
| Value ranges | Impossible values: negative age, rate > 100%, future dates in historical data |

**Common mistakes:**
- Skipping the audit and jumping straight to analysis — silent errors propagate and are hard to trace back
- Using `summary()` alone on the full frame — numeric and character columns summarize very differently; always split by type as shown, or pair with `glimpse()`
- Treating a high null rate as always bad — sometimes nulls are meaningful (e.g. `cancellation_date` is `NA` = not cancelled)

**Three more audit checks for §1:** detect columns that secretly mix types
(rarer in R than pandas, since R columns are type-homogeneous by construction —
but still possible via `list`-columns), measure true memory use, and turn ad-hoc
checks into reusable assertions.

In [ ]:
library(dplyr)
s <- tibble::tibble(
  id       = c(1, 2, 3, 4),
  mixed    = list(1, "two", 3, 4.0),                 # list-column holding mixed types -- R's closest analog to pandas' object-mixing
  platform = c('ios','android','ios','web'),
  revenue  = c(10.0, 20.0, NA, 40.0)
)

# 1) Columns that secretly mix types -- in R this mainly shows up in list-columns,
#    since atomic vector columns are coerced to one type automatically on creation
#    (R's biggest structural difference from pandas' object dtype)
mixed_type_cols <- function(df) {
  out <- list()
  for (c in names(df)) {
    if (is.list(df[[c]])) {
      kinds <- unique(sapply(df[[c]][!sapply(df[[c]], is.null)], function(v) class(v)[1]))
      if (length(kinds) > 1) out[[c]] <- kinds
    }
  }
  out
}
cat('mixed-type columns:', paste(names(mixed_type_cols(s)), collapse = ", "), '\n')

# 2) True memory footprint -- object.size() already counts actual data, not just pointers
print(sapply(s, object.size))

# 3) Reusable validation: promote ad-hoc checks to assertions you re-run after each step
validate <- function(df) {
  stopifnot(
    "id must be unique" = !any(duplicated(df$id)),
    "revenue must be non-negative" = all(df$revenue[!is.na(df$revenue)] >= 0),
    "unexpected platform" = all(df$platform %in% c('ios', 'android', 'web'))
  )
  cat('all checks passed\n')
}
validate(s)

---
## §2 — Fix Types

Wrong types are the silent cause of failed joins, incorrect aggregations, and
broken date math. Fix them immediately after the audit — everything downstream
depends on correct types.

In [ ]:
library(lubridate)

# ── Pattern 1: Date columns loaded as strings ────────────────────────────────
# Symptom: glimpse() shows <chr> for a date column
df <- df |> mutate(event_date = ymd(event_date))                       # explicit format -- fastest + safest
df <- df |> mutate(event_date = parse_date_time(event_date, orders = c("ymd", "mdy", "dmy")))  # auto-detect across formats
df <- df |> mutate(ts = as_datetime(ts / 1000))                        # Unix ms timestamp -> seconds -> datetime

# Verify:
stopifnot("Date parse failed" = inherits(df$event_date, "Date") || inherits(df$event_date, "POSIXct"))

# ── Pattern 2: ID columns loaded as double (signals nulls were present) ──────
# Symptom: user_id shows as <dbl> in glimpse() when it should be a whole number
# Why: unlike pandas, R's integer type CAN hold NA natively -- so this pattern is
# actually rarer in R. It still happens when a column is read as double by readr
# because it saw non-integer-looking text somewhere.
df <- df |> mutate(user_id = as.integer(replace_na(user_id, -1)))   # fill nulls first, then cast
df <- df |> mutate(user_id = as.character(user_id))                 # or keep as character if used as a key

# ── Pattern 3: Boolean columns loaded as int or character ───────────────────
df <- df |> mutate(is_active = as.logical(is_active))                        # from 0/1 int -- R coerces 0/1 -> FALSE/TRUE automatically
df <- df |> mutate(is_active = case_match(is_active,
  "True" ~ TRUE, "False" ~ FALSE, "true" ~ TRUE, "false" ~ FALSE
))

# ── Pattern 4: Numeric columns loaded as character (has non-numeric chars) ──
# Symptom: revenue shows as <chr> -- likely has '$', ',', or '' values
df <- df |> mutate(
  revenue = str_remove_all(revenue, "[\\$,]"),   # strip currency symbols and commas
  revenue = str_trim(revenue),
  revenue = na_if(revenue, ""),                   # empty string -> NA before casting
  revenue = as.double(revenue)
)

# ── Pattern 5: Low-cardinality strings as character -> factor ───────────────
# When: column has < ~50 unique values and appears in group-by/filter frequently
for (col in c('platform', 'region', 'status')) {
  if (col %in% names(df) && n_distinct(df[[col]]) < 50) {
    df[[col]] <- as.factor(df[[col]])
  }
}
# Memory saving: factor stores integers internally with a level lookup table -- same idea as pandas' category dtype

# ── Pattern 6: Cast multiple columns at once ────────────────────────────────
df <- df |> mutate(across(any_of("age"),      as.integer),
                    across(any_of("score"),    as.double),
                    across(any_of("user_id"),  as.character),
                    across(any_of("platform"), as.factor))

**Type-fix decision tree:**

```
Column shows as <chr> in glimpse()?
+-- Looks like a date                -> lubridate::ymd() / parse_date_time()
+-- Has $, %, commas in values       -> str_remove_all() then as.double()
+-- Has True/False/Yes/No strings    -> case_match() then as.logical()
+-- Few unique values (<50)          -> as.factor()
+-- Should be numeric but has blanks -> na_if("") then as.double()

Numeric column shows as double but should be integer?
+-- Fill or drop NAs first, then as.integer()
```

**Common mistakes:**
- `as.integer(col)` when the column has `NA` — unlike pandas (which raises `ValueError`), R's `as.integer()` silently keeps the `NA` and casts everything else; this is *safer* but means a bad cast can go unnoticed without an explicit check
- Parsing ambiguous date formats like `01/02/03` without specifying the order explicitly via `orders =` — `parse_date_time()` guesses among the orders you give it and can guess wrong if the list is too permissive
- Converting ID columns to `integer`/`double` when they have leading zeros (zip codes, product codes) — use `character` instead, identical trap to pandas

**Timezone handling (a §2 type landmine).** A naive timestamp carries no zone;
mixing tz-aware and tz-naive values in a comparison or join raises an error in R
too. `force_tz()`/`with_tz()` is lubridate's pair for *attaching* vs *converting*
a zone — the same conceptual split as pandas' `tz_localize` vs `tz_convert`.

In [ ]:
library(lubridate)
naive <- ymd_hm(c('2024-06-10 14:30', '2024-06-10 18:30'))
cat('naive tz attribute:', tz(naive), '\n')                      # "UTC" -- R always has SOME tz attribute, but treat this as "naive" until deliberately set
aware <- force_tz(naive, 'America/New_York')                      # attach a zone (naive -> aware), does NOT shift the clock time
cat('tz-aware tz       :', tz(aware), '\n')
cat('converted to UTC  :', format(with_tz(aware, 'UTC')), '\n')   # with_tz CHANGES an already-aware time, shifting the clock

---
## §3 — Handle Missing Values

The syntax for filling/dropping `NA`s is in the Tidyverse reference (§3). This
section focuses on the decision: **which strategy to use and when.**

In [ ]:
library(dplyr)
library(tidyr)

# ── Strategy 1: Drop -- when nulls are random and few ────────────────────────
# Use when: null_rate < 5% and nulls appear to be random (not systematic)
# Risk: if nulls are NOT random, dropping introduces bias
df <- df |> drop_na(user_id, event_date)   # only drop on key columns

# ── Strategy 2: Fill with a constant ────────────────────────────────────────
# Use when: null means "zero" or "unknown" by business logic
df <- df |> mutate(
  revenue  = replace_na(revenue, 0),            # null revenue -> 0 spend (not missing, literally zero)
  category = replace_na(category, 'Unknown')    # null category -> explicit Unknown label
)

# ── Strategy 3: Fill with a statistic ───────────────────────────────────────
# Use when: value is missing at random, distribution is roughly symmetric
df <- df |> mutate(
  age      = replace_na(age, median(age, na.rm = TRUE)),       # median -- robust to skew
  score    = replace_na(score, mean(score, na.rm = TRUE)),     # mean -- only if no outliers
  platform = replace_na(platform, names(sort(table(platform), decreasing = TRUE))[1])  # mode -- for categoricals
)

# ── Strategy 4: Group-conditional fill ──────────────────────────────────────
# Use when: the right fill value depends on a grouping variable
# Example: fill missing salary with the median salary for that job title
df <- df |> group_by(job_title) |> mutate(
  salary = replace_na(salary, median(salary, na.rm = TRUE))
) |> ungroup()

# ── Strategy 5: Forward / backward fill ─────────────────────────────────────
# Use when: data is time-ordered and nulls should carry the last known value
df <- df |> arrange(user_id, date) |> group_by(user_id) |> fill(subscription_status, .direction = "down") |> ungroup()

# ── Strategy 6: Flag then fill ──────────────────────────────────────────────
# Use when: the fact that a value is missing is itself informative
# (common in churn models -- missing last_login_date may signal disengagement)
df <- df |> mutate(
  income_missing = as.integer(is.na(income)),                  # 1 = was missing
  income         = replace_na(income, median(income, na.rm = TRUE))  # then fill for modeling
)

# ── Strategy 7: Drop the column entirely ────────────────────────────────────
# Use when: null_rate > 60-70% and no business justification to keep it
threshold <- 0.6
cols_to_drop <- names(df)[colMeans(is.na(df)) > threshold]
df <- df |> select(-any_of(cols_to_drop))
cat(sprintf('Dropped %d columns: %s\n', length(cols_to_drop), paste(cols_to_drop, collapse = ", ")))

**Strategy selection guide:**

| Null rate | Null mechanism | Recommended strategy |
| :--- | :--- | :--- |
| < 5% | Random | Drop the rows |
| Any | Business-defined zero | Fill with 0 or 'Unknown' |
| Any | Time-ordered carry-forward | Forward fill (sort first) |
| Any | Depends on group | Group-conditional fill |
| Any | Missingness is informative | Flag + fill |
| > 60% | Any | Drop the column |

**Common mistakes:**
- Filling with the global mean/median when group values differ significantly — always check if group-conditional fill (§3 Strategy 4) is more appropriate
- Forward-filling without `arrange()`-ing by date first — fills in row order, not time order, producing wrong values, the exact same trap as pandas
- Dropping rows with any `NA` (`drop_na()` with no arguments) when only a few columns matter — pass specific column names to be precise
- Forgetting `na.rm = TRUE` when computing the fill statistic itself (`median()`, `mean()`) — without it, the statistic you're filling with is itself `NA`

---
## §4 — Handle Duplicates

Two distinct types of duplicates — full row duplicates (data pipeline bugs) and
key-level duplicates (fan-out from joins or intentional multi-row data). They
require different fixes.

In [ ]:
library(dplyr)

# ── Detect duplicates ────────────────────────────────────────────────────────
# Full duplicates -- every column is identical
n_full <- sum(duplicated(df))
cat(sprintf('Full duplicates: %d (%.1f%%)\n', n_full, n_full / nrow(df) * 100))

# Key-level duplicates -- same key but potentially different other columns
key <- c('user_id', 'order_id')
n_key <- sum(duplicated(df[key]))
cat('Key-level duplicates:', n_key, '\n')

# Inspect the duplicated records before acting
dup_mask <- duplicated(df[key]) | duplicated(df[key], fromLast = TRUE)   # flags ALL copies, both directions
df |> filter(dup_mask) |> arrange(across(all_of(key))) |> head(20)        # look at them side by side

# ── Fix 1: Drop full duplicates ──────────────────────────────────────────────
# Use when: rows are identical -- clearly a pipeline or loading bug
df <- df |> distinct()

# ── Fix 2: Keep first or last occurrence ─────────────────────────────────────
# Use when: one record per key is correct, duplicates are re-submissions or updates
df <- df |> arrange(desc(updated_at))                       # latest record first
df <- df |> distinct(user_id, .keep_all = TRUE)              # keep the most recent

# ── Fix 3: Aggregate to resolve ─────────────────────────────────────────────
# Use when: duplicates exist because one user has multiple rows that should be summed
df <- df |> group_by(user_id) |> summarise(
  total_spend = sum(spend, na.rm = TRUE),
  visit_count = n(),
  last_seen   = max(date),
  .groups = "drop"
)

# ── Fix 4: Diagnose why duplicates exist ────────────────────────────────────
# Check if duplication is caused by a join fanout
# Before join:
cat('Users before join:', n_distinct(df_users$user_id), '\n')
merged <- df_users |> left_join(df_orders, by = "user_id")
cat('Users after join: ', n_distinct(merged$user_id), '\n')
# If these differ, the right table has multiple rows per user_id
# Check the right table:
print(summary(df_orders |> count(user_id) |> pull(n)))

**Duplicate type decision tree:**

```
Are all columns identical?
+-- Yes -> data pipeline bug -> distinct()
+-- No  -> key-level duplicates -> inspect why
         +-- One should be the "truth" (latest update)  -> arrange + distinct(.keep_all=TRUE)
         +-- All rows have valid data (transactions)     -> aggregate (sum, count, max via summarise)
         +-- Caused by a join                            -> fix the join, not the output
```

**Common mistakes:**
- Dropping duplicates before understanding why they exist — if caused by a bad join, the fix is upstream, not a `distinct()` bandage
- Using `distinct(.keep_all = TRUE)` without `arrange()`-ing first — "first" means first in current row order, which may be arbitrary, same trap as pandas' unsorted `keep='first'`
- Checking only full duplicates and missing key-level duplicates — the more dangerous type is usually the key-level one

---
## §5 — Fix String & Categorical Inconsistencies

The same real-world entity appearing under different labels is one of the most
common data quality issues. `group_by()` and `*_join()` operations silently fail
to match when `"US"` and `"us"` are treated as different groups.

In [ ]:
library(dplyr)
library(stringr)

# ── Step 1: Always inspect the raw unique values first ──────────────────────
table(df$country, useNA = "ifany")
# Look for: trailing spaces, mixed case, abbreviations vs full names, typos

# ── Fix 1: Normalize case and whitespace -- do this first, always ────────────
df <- df |> mutate(country = str_to_lower(str_trim(country)))
# Before: 'US', ' us', 'US ', 'us' -> After: 'us' (all unified)

# ── Fix 2: Map known variants to a canonical value ──────────────────────────
df <- df |> mutate(country = case_match(country,
  'us'             ~ 'United States',
  'usa'            ~ 'United States',
  'united states'  ~ 'United States',
  'u.s.'           ~ 'United States',
  'uk'             ~ 'United Kingdom',
  'united kingdom' ~ 'United Kingdom',
  'gb'             ~ 'United Kingdom',
  .default = country     # preserve values not in the map (don't silently nullify them) -- R's coalesce-with-self pattern
))

# ── Fix 3: Replace known typos ──────────────────────────────────────────────
df <- df |> mutate(country = case_match(country,
  'cnaada'  ~ 'canada',
  'germny'  ~ 'germany',
  'brazill' ~ 'brazil',
  .default = country
))

# ── Fix 4: Bucket sparse categories into 'Other' ───────────────────────────
# Use when: long tail of rare categories that don't merit their own group
top_countries <- df |> count(country, sort = TRUE) |> slice_head(n = 10) |> pull(country)
df <- df |> mutate(country_grouped = if_else(country %in% top_countries, country, "Other"))

# ── Fix 5: Standardize boolean-like categoricals ─────────────────────────────
# Common in survey or CRM data
df <- df |> mutate(subscribed = case_match(str_to_lower(str_trim(subscribed)),
  'yes' ~ 1L, 'no' ~ 0L,
  'y'   ~ 1L, 'n'  ~ 0L,
  '1'   ~ 1L, '0'  ~ 0L,
  'true' ~ 1L, 'false' ~ 0L
))

# ── Fix 6: Enforce a controlled vocabulary ──────────────────────────────────
# Catch unexpected values after cleaning
valid_platforms <- c('ios', 'android', 'web', 'other')
unexpected <- setdiff(unique(df$platform), valid_platforms)
if (length(unexpected) > 0) {
  cat('WARNING: unexpected platform values:', paste(unexpected, collapse = ", "), '\n')
}

**Cleaning sequence — always in this order:**

1. `str_trim()` — remove whitespace
2. `str_to_lower()` — normalize case
3. `case_match(..., .default = col)` for typos — fix known typos
4. `case_match(..., .default = col)` for synonyms — unify synonyms (same function, different lookup; R doesn't need two separate verbs)
5. Bucket rare values into `'Other'`
6. Validate against controlled vocabulary

**Common mistakes:**
- Using `case_match()` **without** `.default = col` — unmatched values become `NA`, silently nullifying valid data, the exact same trap as pandas' `.map()` without `.fillna(df['col'])`
- Normalizing case after mapping — map first on lowercased values, not on mixed-case raw values
- Forgetting that `case_match()` matches **exact** values only; `str_detect()`/`str_replace()` use substring/regex matching — they are different tools, same split as pandas' `.replace()` vs `.str.replace()`

---
## §6 — Handle Outliers

Outliers are not always errors. The decision is: is this value impossible,
implausible, or just extreme? Each case requires a different response.

In [ ]:
library(dplyr)

# ── Step 1: Detect outliers ──────────────────────────────────────────────────

# Method A: IQR -- robust, distribution-free
Q1  <- quantile(df$revenue, 0.25, na.rm = TRUE)
Q3  <- quantile(df$revenue, 0.75, na.rm = TRUE)
IQR_ <- Q3 - Q1
lower <- Q1 - 1.5 * IQR_
upper <- Q3 + 1.5 * IQR_
outliers_iqr <- df |> filter(revenue < lower | revenue > upper)
cat(sprintf('IQR outliers: %d rows\n', nrow(outliers_iqr)))

# Method B: Z-score -- works well for roughly normal distributions
# R's filter() already drops NA comparisons cleanly (treats them as non-matching),
# so there is no pandas-style "Item wrong length" trap to work around here.
z <- (df$revenue - mean(df$revenue, na.rm = TRUE)) / sd(df$revenue, na.rm = TRUE)
outliers_z <- df |> filter(abs(z) > 3)
cat(sprintf('Z-score outliers (|z|>3): %d rows\n', nrow(outliers_z)))

# Method C: Domain threshold -- clearest when business rules exist
outliers_domain <- df |> filter(age < 0 | age > 120)

# ── Step 2: Decide what to do ───────────────────────────────────────────────

# Treatment A: Remove -- only if value is impossible or a clear data error
df <- df |> filter(age >= 0 & age <= 120)

# Treatment B: Winsorize (cap) -- clip to plausible bounds, keep the row
# Use when: extreme but real values exist (e.g. one user spent $1M)
p01 <- quantile(df$revenue, 0.01, na.rm = TRUE)
p99 <- quantile(df$revenue, 0.99, na.rm = TRUE)
df <- df |> mutate(revenue_capped = pmin(pmax(revenue, p01), p99))   # R's clip() equivalent: nested pmin/pmax

# Treatment C: Flag and keep -- preserve the outlier, mark it for downstream
# Use when: the outlier may be real and important (e.g. a whale customer)
df <- df |> mutate(is_revenue_outlier = as.integer(revenue > upper))

# Treatment D: Log transform -- compress the scale, don't remove
# Use when: data is right-skewed (common for spend, income, counts)
df <- df |> mutate(log_revenue = log1p(revenue))   # log1p = log(x+1), handles zeros safely, same function name as NumPy

# ── Summary: check the effect of your treatment ─────────────────────────────
summary(df$revenue)
summary(df$revenue_capped)

**Treatment decision guide:**

| Outlier type | Treatment | Reason |
| :--- | :--- | :--- |
| Impossible value (age = -5) | Remove | Data entry error — the row is invalid |
| Plausible but extreme (spend = $1M) | Winsorize or flag | Real event — removing distorts the picture |
| Right-skewed distribution | Log transform | Compresses scale without losing data |
| Extreme but informative (whale user) | Flag + keep | May be the most important segment |

**Common mistakes:**
- Removing outliers reflexively — always ask "is this impossible or just unexpected?"
- Using Z-score on heavily skewed data — Z-score assumes normality; IQR is more robust for skewed distributions
- Winsorizing before splitting train/test — compute the percentile bounds on train set only, then apply to test to avoid leakage
- Using `log()` on columns with zeros — produces `-Inf` with a warning; always use `log1p()`, base R's name for the same function as NumPy's `np.log1p()`

---
## §7 — Clean Column Names

Messy column names break `filter()`/`mutate()` with non-syntactic names (requiring
backtick-quoting everywhere) and SQL integrations. Fix them once, right after
loading. The `janitor` package has a one-call solution; the manual pipeline below
shows what it does internally.

In [ ]:
library(stringr)
library(dplyr)

# ── The standard cleaning pipeline ──────────────────────────────────────────
clean_column_names <- function(df) {
  new_names <- names(df) |>
    str_trim() |>                          # remove leading/trailing whitespace
    str_to_lower() |>                      # lowercase everything
    str_remove_all("[^\\w\\s]") |>          # remove special chars (keep letters, digits, _)
    str_replace_all("\\s+", "_") |>         # spaces -> underscores
    str_replace_all("_+", "_") |>           # collapse multiple underscores
    str_remove("^_+") |> str_remove("_+$")  # remove leading/trailing underscores
  names(df) <- new_names
  df
}

df <- clean_column_names(df)

# Example transformation:
# ' Total Revenue ($) ' -> 'total_revenue'   (trailing _ removed by the final str_remove)
# 'User ID'             -> 'user_id'
# 'Q1 2024 Sales'       -> 'q1_2024_sales'
# '__internal__'        -> 'internal'

# Or, equivalently, with the janitor package (the production shortcut for the above):
# df <- janitor::clean_names(df)

# ── Handle duplicate column names after cleaning ─────────────────────────────
# Can happen when 'Revenue $' and 'Revenue %' both become 'revenue'
nm <- names(df)
for (dup in unique(nm[duplicated(nm)])) {
  idx <- which(nm == dup)
  nm[idx] <- paste0(dup, "_", seq_along(idx) - 1)
}
names(df) <- nm

# ── Rename specific columns with a map ──────────────────────────────────────
df <- df |> rename(
  customer_id = cust_id,
  revenue     = rev,
  event_date  = dt
)

# ── Validate final column names ──────────────────────────────────────────────
bad_cols <- names(df)[!str_detect(names(df), "^[a-z][a-z0-9_]*$")]
if (length(bad_cols) > 0) {
  cat('WARNING: non-standard column names remain:', paste(bad_cols, collapse = ", "), '\n')
}

**Common mistakes:**
- Cleaning column names in the middle of a pipeline — do it immediately after loading, before any selection or filtering
- Forgetting to handle duplicate column names after cleaning — two different columns silently collapse to the same name (R will actually append `...1`/`...2` automatically on `tibble` creation in some cases, but not after a manual `names<-` assignment like above — always check explicitly)
- Using `str_replace_all(" ", "_")` without collapsing multiple spaces first — `'Total  Revenue'` becomes `'Total__Revenue'`, identical trap to pandas

---
## Decision Guide

```
Just loaded a new dataset?
\-- Always run: dim -> glimpse() -> null summary -> duplicates -> summary() -> value ranges  (§1)

Column type is wrong?
+-- Date stored as string             -> lubridate::ymd() / parse_date_time()           (§2)
+-- ID stored as double               -> fill NAs -> as.integer() or as.character()     (§2)
+-- Number has $, commas              -> str_remove_all() -> as.double()                (§2)
\-- String with <50 unique values     -> as.factor()                                    (§2)

Column has missing values?
+-- Null rate < 5%, random            -> drop_na(col1, col2, ...)                       (§3)
+-- Null means zero by logic          -> replace_na(x, 0)                               (§3)
+-- Null should carry forward         -> arrange first -> group_by() |> fill()          (§3)
+-- Null depends on group             -> group_by() |> mutate(replace_na(...))          (§3)
+-- Null is informative               -> flag column + fill                             (§3)
\-- Null rate > 60%                   -> drop the column                                (§3)

Duplicate rows detected?
+-- All columns identical             -> distinct()                                     (§4)
+-- Same key, need latest record      -> arrange by date -> distinct(.keep_all=TRUE)    (§4)
+-- Same key, all rows are valid      -> group_by() |> summarise()                      (§4)
\-- Caused by a join                  -> fix the join upstream                          (§4)

Category labels are inconsistent?
\-- trim -> lower -> replace typos -> map synonyms -> bucket rare values               (§5)

Outliers detected?
+-- Impossible value                  -> remove the row                                 (§6)
+-- Plausible but extreme             -> pmin/pmax() to winsorize                       (§6)
+-- Right-skewed distribution         -> log1p() transform                              (§6)
\-- Extreme but meaningful            -> flag column and keep                           (§6)

Column names are messy?
\-- trim -> lower -> special chars -> spaces->underscore -> deduplicate                (§7)
```